# Interactive research: trading the ETH/BTC ratio

A scratchpad for exploring the ETH/BTC perpetual spread with screamer. It reuses
the tested functions in `deribit_pairs_backtest.py`, so this notebook stays a thin
interactive layer: load data, look at the spread, ask whether it mean-reverts, and
backtest a fade under taker and maker execution.

Run it from the `examples/` directory. It defaults to the small committed sample
in `../devtools/data/`; for real research download a few days first:

```
python deribit_download.py --days 3 --out-dir .
```
and set `DATA_DIR = "."` below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import deribit_pairs_backtest as bt
from screamer import RollingZscore, RollingOU

DATA_DIR = "../devtools/data"          # a folder with *ETH-PERPETUAL*.csv and *BTC-PERPETUAL*.csv
BASE, QUOTE = "ETH-PERPETUAL", "BTC-PERPETUAL"

## 1. Load and build the log-ratio spread

In [ ]:
BAR = "60s"
times, spread, half = bt.load_spread(DATA_DIR, BASE, QUOTE, BAR)
print(f"{len(spread)} bars of {BAR}, estimated half-spread {half:.2f} bps (both legs)")

## 2. Look at it: ratio, z-score, and the mean-reversion rate

In [ ]:
z = np.asarray(RollingZscore(100)(spread))
rate = np.asarray(RollingOU(200, output=0)(spread))
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(times, np.exp(spread)); ax[0].set_ylabel("ETH/BTC ratio")
ax[1].plot(times, z, lw=0.7); ax[1].axhline(0, color="k", lw=0.5); ax[1].set_ylabel("z-score")
ax[2].plot(times, rate, lw=0.7); ax[2].axhline(np.nanmedian(rate), color="r", ls=":")
ax[2].set_ylabel("OU rate"); ax[2].set_xlabel("time")
plt.tight_layout()

## 3. Is there mean-reversion? Return autocorrelation across timeframes

Negative lag-1 autocorrelation of the ratio's returns means it reverts. On a
liquid pair this is usually true at every timeframe (bid-ask bounce plus real
reversion), which is why fading is *gross* profitable.

In [ ]:
for b in ["1s", "5s", "15s", "60s", "300s"]:
    _, s, _ = bt.load_spread(DATA_DIR, BASE, QUOTE, b)
    ac = pd.Series(np.diff(s)).autocorr(1)
    print(f"{b:>5}: {len(s):>7} bars   lag-1 autocorr {ac:+.3f}")

## 4. Backtest a fade, taker vs maker

The fade shorts the ratio when the z-score is high and longs it when low, with a
regime gate that stands aside when `RollingOU` says the ratio is trending. The
cost per trade is the crux: a taker pays both fees plus the crossed spread; a
maker earns a fraction (`capture`) of the spread instead.

In [ ]:
def backtest(bar="30s", z_window=100, regime_window=200, entry=1.0, exit=0.25,
             taker_fee=5.0, maker_fee=0.0, capture=0.5):
    times, spread, half = bt.load_spread(DATA_DIR, BASE, QUOTE, bar)
    sec = pd.Timedelta(bar).total_seconds()
    bpy = 365 * 24 * 3600 / sec
    minutes = len(spread) * sec / 60
    taker = bt.turnover_cost_bps("taker", taker_fee, maker_fee, half, capture)
    maker = bt.turnover_cost_bps("maker", taker_fee, maker_fee, half, capture)
    pos = bt.fade_positions(spread, z_window, entry, exit, regime_window)
    res = {k: bt.evaluate(spread, pos, c, bpy)
           for k, c in [("gross", 0.0), ("taker", taker), ("maker", maker)]}
    per_trade = minutes / max(res["gross"]["trades"], 1)
    return dict(times=times, spread=spread, pos=pos, res=res,
                taker=taker, maker=maker, half=half, bpy=bpy, per_trade=per_trade)

b = backtest(bar="30s", entry=1.0, exit=0.25)
print(f"taker cost {b['taker']:+.2f} bps, maker cost {b['maker']:+.2f} bps, "
      f"~{b['per_trade']:.0f} min/trade")
for k in ["gross", "taker", "maker"]:
    r = b["res"][k]
    print(f"  {k:6s} Sharpe {r['sharpe']:7.1f}   total {r['total']*100:+.2f}%")

In [ ]:
eq = lambda cost: np.cumsum(bt.evaluate(b["spread"], b["pos"], cost, b["bpy"])["net"]) * 100
t = b["times"][1:]
plt.figure(figsize=(11, 5))
plt.plot(t, eq(b["maker"]), color="#28a745", label="maker")
plt.plot(t, eq(0.0), color="#888", label="gross")
plt.plot(t, eq(b["taker"]), color="#d62728", ls="--", label="taker")
plt.axhline(0, color="k", lw=0.5); plt.legend(); plt.grid(alpha=0.25)
plt.ylabel("cumulative P&L (%)"); plt.title("ETH/BTC fade: taker vs maker")

## 5. How much maker capture do you need to break even?

`capture` is the fraction of the half-spread a resting order actually earns after
missed fills and adverse selection. Without an order book we cannot measure it, so
sweep it: the point where P&L crosses zero is the execution quality you need.

In [ ]:
for cap in [0.0, 0.2, 0.3, 0.5, 0.7, 1.0]:
    c = bt.turnover_cost_bps("maker", 5.0, 0.0, b["half"], cap)
    r = bt.evaluate(b["spread"], b["pos"], c, b["bpy"])
    print(f"capture {cap:.1f}  cost {c:+.2f} bps  total {r['total']*100:+6.2f}%  Sharpe {r['sharpe']:6.1f}")

## 6. Frequency vs edge: sweep the entry threshold

In [ ]:
print(f"{'entry':>6} {'min/trade':>10} {'maker%':>9} {'taker%':>9}")
for entry in [0.5, 0.75, 1.0, 1.5, 2.0]:
    r = backtest(bar="30s", entry=entry, exit=entry * 0.25)
    print(f"{entry:>6.2f} {r['per_trade']:>10.1f} "
          f"{r['res']['maker']['total']*100:>9.2f} {r['res']['taker']['total']*100:>9.2f}")

## Notes and caveats

- **Maker capture is the load-bearing assumption.** The trade CSVs have no order
  book, so fills are approximated by a fixed fraction of the spread. Treat the
  capture sweep in section 5 as the real question: can you actually earn that
  spread live?
- **Buy & hold can beat the fade over a short window** simply because one leg
  drifted against the other. That is directional luck, not a spread edge.
- **Three days is not a sample.** Download more and re-run before trusting any
  number here.
- Fees are placeholders; set your actual Deribit tier. This is research, not
  trading advice.